# 🔌 MCP — kết nối agent với công cụ ngoài (demo: CSDL SQLite)

> Demo nâng cao cho lớp GenAI.

## MCP là gì?

**Model Context Protocol (MCP)** là một chuẩn mở để agent dùng *công cụ/dữ liệu* từ một **server** bên ngoài
(CSDL, hệ thống file, API...). Lợi ích: viết tool **một lần** trong MCP server, rồi **mọi** ứng dụng/agent
(Claude Desktop, Cursor, LangChain...) đều dùng được — không phải nhúng cứng tool vào từng app.

Kiến trúc gồm 2 phía, chạy ở **2 tiến trình riêng**, nói chuyện qua giao thức MCP:

```
[Notebook / Agent]  ──(MCP, stdio)──►  [MCP server: travel_mcp_server.py]
   langchain-mcp-adapters                FastMCP + SQLite
```

Trong notebook này ta sẽ:
1. Viết một **MCP server** nhỏ (FastMCP) bọc một CSDL SQLite về du lịch.
2. Dùng `langchain-mcp-adapters` để **nạp tool của server** thành tool LangChain.
3. Đưa cho `create_agent` để agent **tự truy vấn CSDL** trả lời câu hỏi tiếng tự nhiên.

## 1. Cài đặt & import

`uv sync` đã cài sẵn `langchain-mcp-adapters` và `mcp`. Nếu dùng pip:
```
%pip install langchain-mcp-adapters mcp
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import os
import sys
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler
from IPython.display import Markdown, display

load_dotenv()
print("Imports OK")

## 2. LLM + Langfuse (giống các notebook khác)

In [ ]:
# --- Cấu hình ---
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.7

# --- LLM ---
llm = init_chat_model(model=MODEL_NAME, temperature=TEMPERATURE)

# --- Langfuse: phải khởi tạo global client trước, rồi tạo CallbackHandler ---
_host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=_host,
)
langfuse_handler = CallbackHandler()

try:
    if get_client().auth_check():
        print("✅ Langfuse đã kết nối:", _host)
except Exception:
    print("⚠️  Chưa kết nối Langfuse (kiểm tra key/host trong .env). Agent vẫn chạy, chỉ là không có trace.")

print("Model:", MODEL_NAME)

## 3. Viết MCP server (FastMCP + SQLite)

Cell dưới dùng `%%writefile` để **ghi server ra file** `travel_mcp_server.py` (vì server phải chạy ở
tiến trình riêng). Server tạo sẵn một CSDL SQLite về điểm đến du lịch và mở 3 tool:
- `list_tables()` — liệt kê bảng
- `describe_table(table)` — xem cột (để agent biết tên cột thật)
- `run_query(sql)` — chạy câu `SELECT`; nếu lỗi thì trả `{'error': ...}` để agent **tự sửa**, không crash.

In [ ]:
%%writefile travel_mcp_server.py
import sqlite3
from mcp.server.fastmcp import FastMCP

DB_PATH = "travel_demo.db"

def init_db():
    con = sqlite3.connect(DB_PATH); cur = con.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS destinations(
        city TEXT, country TEXT, best_season TEXT, avg_daily_budget_usd INTEGER, famous_for TEXT)""")
    if cur.execute("SELECT COUNT(*) FROM destinations").fetchone()[0] == 0:
        cur.executemany("INSERT INTO destinations VALUES (?,?,?,?,?)", [
            ("Tokyo","Japan","Spring",120,"ẩm thực, văn hoá, hoa anh đào"),
            ("Paris","France","Autumn",140,"nghệ thuật, kiến trúc, ẩm thực"),
            ("Bali","Indonesia","Dry season",60,"biển, nghỉ dưỡng"),
            ("New York","USA","Autumn",180,"đô thị, bảo tàng, ẩm thực"),
            ("Bangkok","Thailand","Winter",45,"ẩm thực đường phố, đền chùa"),
        ])
    con.commit(); con.close()

mcp = FastMCP("travel-db")

@mcp.tool()
def list_tables() -> list[str]:
    """Liệt kê tên các bảng trong CSDL."""
    con = sqlite3.connect(DB_PATH)
    rows = con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    con.close(); return [r[0] for r in rows]

@mcp.tool()
def describe_table(table: str) -> list[dict]:
    """Trả về tên cột và kiểu dữ liệu của một bảng (xem trước khi viết query)."""
    con = sqlite3.connect(DB_PATH)
    rows = con.execute(f"PRAGMA table_info({table})").fetchall()
    con.close(); return [{"column": r[1], "type": r[2]} for r in rows]

@mcp.tool()
def run_query(sql: str) -> list[dict]:
    """Chạy một câu SELECT (chỉ đọc). Nếu lỗi, trả {'error': ...} để bạn sửa lại."""
    if not sql.strip().lower().startswith("select"):
        return [{"error": "Chỉ cho phép câu lệnh SELECT."}]
    con = sqlite3.connect(DB_PATH); con.row_factory = sqlite3.Row
    try:
        return [dict(r) for r in con.execute(sql).fetchall()]
    except Exception as e:
        return [{"error": str(e)}]
    finally:
        con.close()

if __name__ == "__main__":
    init_db()
    mcp.run()  # mặc định chạy qua stdio

## 4. Kết nối tới MCP server và nạp tool

`MultiServerMCPClient` khởi chạy server như một tiến trình con (`stdio`) rồi nạp tool của nó.
`sys.executable` = Python của `.venv` hiện tại (đã có sẵn `mcp`).

In [ ]:
##### TODO: Thực hành #####
# Yêu cầu:
#   - Tạo MultiServerMCPClient kết nối tới server tên "travel":
#       command   = sys.executable          # python của .venv
#       args      = ["travel_mcp_server.py"] # file MCP server đã ghi ở trên
#       transport = "stdio"
#   - get_tools() là hàm async -> dùng `await client.get_tools()` để nạp tool, lưu vào `tools`.
#######################
### START CODE HERE ###
#######################


##### End TODO #####

print("Tool nạp từ MCP server:", [t.name for t in tools])

## 5. Đưa tool MCP cho agent

Agent được nhắc: xem schema trước (list_tables/describe_table) rồi mới viết `SELECT` — để dùng đúng tên cột.

In [ ]:
SYSTEM = (
    "Bạn là trợ lý truy vấn CSDL du lịch. Khi cần dữ liệu: trước tiên dùng list_tables/describe_table "
    "để biết bảng và CỘT có thật, sau đó viết câu SELECT đúng tên cột. Trả lời ngắn gọn bằng tiếng Việt."
)

##### TODO: Thực hành #####
# Yêu cầu:
#   - Tạo agent bằng create_agent(...) với: llm, danh sách `tools` vừa nạp từ MCP server,
#     và system_prompt=SYSTEM. Gán vào biến `agent`.
#######################
### START CODE HERE ###
#######################


##### End TODO #####

print("✅ Agent đã sẵn sàng với", len(tools), "tool từ MCP")

## 6. Hỏi đáp — agent tự truy vấn CSDL

`agent.ainvoke(...)` cũng là async. Agent sẽ tự gọi các tool MCP (xem schema → SELECT) để trả lời.

In [ ]:
async def ask(question: str):
    res = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]},
        config={"callbacks": [langfuse_handler]},
    )
    print("👤", question)
    print("🤖", res["messages"][-1].content, "\n")

await ask("Thành phố nào có chi phí mỗi ngày thấp nhất, và nổi tiếng về gì?")
await ask("Liệt kê các điểm đến nên đi vào mùa thu (Autumn).")
await ask("Ngân sách 5 ngày ở Tokyo cho 2 người khoảng bao nhiêu USD?")

## 7. Dùng MCP server CÓ SẴN (không cần tự viết)

Điểm mạnh của MCP: hàng trăm server có sẵn. Chỉ cần đổi cấu hình `command/args` là agent dùng được ngay.
Vài ví dụ (cần Node hoặc uv):

```python
MultiServerMCPClient({
    # CSDL SQLite có sẵn (Python, qua uvx):
    "sqlite": {"command": "uvx", "args": ["mcp-sqlite", "duong_dan.db"], "transport": "stdio"},

    # Hệ thống file (Node, qua npx):
    "fs": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", "/duong/dan"],
           "transport": "stdio"},
})
```

Phần code nạp tool + đưa cho `create_agent` **y hệt** như trên — đó chính là giá trị của chuẩn MCP.

## 8. Xem trace trên Langfuse

Mỗi câu hỏi sẽ thành một trace, thấy rõ agent gọi tool MCP nào (list_tables → describe_table → run_query).

In [ ]:
get_client().flush()
host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL") or "https://cloud.langfuse.com"
print("Mở dashboard Langfuse:", host)

## 9. Tổng kết — MCP

- **MCP** tách *công cụ* (server) khỏi *ứng dụng* (agent): viết tool một lần, mọi app dùng chung.
- `langchain-mcp-adapters` nạp tool MCP thành tool LangChain → ghép thẳng vào `create_agent`.
- Server chạy ở **tiến trình riêng**, nói chuyện qua `stdio` (hoặc HTTP).
- Đổi từ server tự viết sang server có sẵn (`uvx mcp-sqlite`, `npx ...`) chỉ là đổi `command/args`.

> Gợi ý mở rộng: ghép tool MCP này vào graph Orchestrator (notebook 02) để agent vừa lập kế hoạch
> vừa tra CSDL thật.